# 05. Reading Text (OCR) · QR Codes

> OCR runs **two** models in a row: ① find **where** the text is, and ② **read** the text in each spot.
> `easyocr` bundles the two together. QR codes are not read by AI, but by fixed rules (image processing).

### What you will learn
1. The shape of an easyocr result: (4 corner points, text, confidence)
2. Filtering by confidence, and setting the **reading order** (top→bottom, left→right)
3. Making and reading QR codes, and the difference between AI and rule-based methods

| | Find text | Read text |
|---|---|---|
| Model inside easyocr | CRAFT (map of text areas) | CRNN (reads one line left→right) |
| Result | tilted rectangles | the text inside + confidence |

easyocr runs on the **CPU** with PyTorch (not OpenVINO).

## 1. Setup

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us); this notebook is in notebooks/en/
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

## 2. Photo — take it so the text is visible (a book cover, a notice, etc.)

In [ ]:
PHOTO = "sample.jpg"          # photo to use when there is no webcam (in the same folder as this notebook)

def show(bgr, width=720):
    """Show an OpenCV photo (BGR) inside the notebook."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """Take one frame from the webcam. If there is no webcam, use the PHOTO file."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # some webcams give dark first frames
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"No webcam and no {PHOTO} — put a photo file name in PHOTO"
        print("No webcam, so using the file:", PHOTO)
    return frame

frame = grab()
print("Photo size (height, width, channels):", frame.shape)
show(frame)

## 3. Reading text — easyocr
- `Reader(["ko", "en"])` — Korean + English. It reads both, so English text works too. More languages make it slower.
- `download_enabled=False` — it does not go to the internet to download models. The models are already in `models/code/easyocr`.
- Pass the photo **in grayscale**: the color-photo path gives errors on some Windows PCs, so edge-lab also passes grayscale. Reading text does not need color.

In [ ]:
import easyocr

OCR_DIR = MODELS / "code" / "easyocr"
t0 = time.perf_counter()
reader = easyocr.Reader(["ko", "en"], gpu=False,           # language codes: Korean + English (the models in OCR_DIR are for these two)
                        model_storage_directory=str(OCR_DIR), user_network_directory=str(OCR_DIR),
                        download_enabled=False)              # do not download from the internet
print(f"Loaded in {time.perf_counter() - t0:.1f} s")

grey = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
t0 = time.perf_counter()
results = reader.readtext(grey)                             # [(4 corner points, text, confidence), ...]
print(f"{len(results)} lines · {(time.perf_counter() - t0) * 1000:.0f} ms")
for quad, text, conf in results:
    print(f"{conf:.2f}  {text}")

In [ ]:
view = frame.copy()
for quad, text, conf in results:
    pts = np.array(quad, dtype=np.int32)
    color = (0, 160, 0) if conf >= 0.5 else (0, 0, 255)      # low confidence in red
    cv2.polylines(view, [pts], True, color, 2)               # each text spot is a tilted rectangle (4 corner points)
show(view)

## 4. Filter · join in reading order
The order easyocr returns may not be the reading order. We put it in the order a person reads:
1. Throw away results with low confidence (e.g. below 0.5).
2. Split into lines by the **center y** of each rectangle — if the y values are within half the text height, they are on the same line.
3. Inside the same line, sort by **x**.

In [ ]:
def read_order(results, min_conf=0.5):
    items = []
    for quad, text, conf in results:
        if conf < min_conf:
            continue
        q = np.array(quad, float)
        items.append({"x": q[:, 0].min(), "cy": q[:, 1].mean(), "h": q[:, 1].max() - q[:, 1].min(), "text": text})
    items.sort(key=lambda it: it["cy"])
    lines, cur = [], []
    for it in items:
        if cur and abs(it["cy"] - cur[-1]["cy"]) > max(it["h"], cur[-1]["h"]) / 2:   # move on to the next line
            lines.append(cur); cur = []
        cur.append(it)
    if cur:
        lines.append(cur)
    return "\n".join(" ".join(it["text"] for it in sorted(line, key=lambda it: it["x"])) for line in lines)

print(read_order(results))

## 5. QR — make and read
A QR code finds its position with the big squares in three corners (finder patterns), then reads the black and white cells by fixed rules. Thanks to **error correction**, it can still be read when part of it is covered.
It does not "learn" like AI — it calculates by the rules, so the result is either right or wrong (there is no confidence).

In [ ]:
enc = cv2.QRCodeEncoder.create()
qr = enc.encode("https://example.com/edge-lab")             # a small black-and-white image (one cell = 1 pixel)
print("QR cells:", qr.shape)
qr = cv2.resize(qr, (qr.shape[1] * 8, qr.shape[0] * 8), interpolation=cv2.INTER_NEAREST)   # cells 8x bigger — NEAREST so it does not blur
qr = cv2.copyMakeBorder(qr, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=255)               # white margin around it (the quiet zone)
qr_bgr = cv2.cvtColor(qr, cv2.COLOR_GRAY2BGR)
show(qr_bgr, 240)

def read_qr(bgr):
    ok, texts, pts, _ = cv2.QRCodeDetector().detectAndDecodeMulti(bgr)
    if not ok or pts is None:
        return []
    return [(t, p.astype(int)) for t, p in zip(texts, pts) if t]

print("QR we made:", [t for t, _ in read_qr(qr_bgr)])
print("Webcam photo:", [t for t, _ in read_qr(frame)] or "no QR")

In [ ]:
# Can it still be read when covered? — cover a bit more of the center each time
for frac in (0.0, 0.05, 0.1, 0.2):
    t = qr_bgr.copy()
    hh, ww = t.shape[:2]
    s = int(min(hh, ww) * frac)
    cv2.rectangle(t, (ww // 2 - s // 2, hh // 2 - s // 2), (ww // 2 + s // 2, hh // 2 + s // 2), (0, 0, 0), -1)
    print(f"center {frac:.0%} covered → {[x for x, _ in read_qr(t)] or 'cannot read'}")

## Common errors
| Symptom | Cause · Fix |
|---|---|
| `Missing ... model` · tries to download | `models/code/easyocr` is missing → `setup.bat` |
| `too many values to unpack` | You passed a color photo → use grayscale |
| Reads the text wrong | Too small or blurry → move closer, add light. Filter by confidence |
| Cannot read the QR | Too small, tilted, or no white margin |

## Summary
- OCR = a find model + a read model. The result is a list of (rectangle, text, confidence), and you set the reading order yourself.
- Use confidence to decide "whether to throw it away".
- QR is rule-based — within the error-correction range, it can be read even when covered.

## Try it
1. What do `reader.readtext(grey, detail=0)` and `paragraph=True` return?
2. Make the photo blurry (`cv2.GaussianBlur`) or small (`cv2.resize`). At what point can it no longer read?
3. Make a QR with non-English text (for example Korean) in it, and read it back.
4. From a receipt photo, pull out only the numbers on the "Total" line.
   <details><summary>Hint</summary>

   Among the lines from `read_order(...)`, find the line that contains "Total", then take the numbers out with `re.findall(r"[\d,]+", line)`.
   </details>

---
# 🛠 Mini project — Business Card Scanner

**What to build**: Read the text from a photo of a business card (or a notice), pick out **phone number · email · name candidate**, then make a **QR** with the contact and save it.
If you scan the QR with a phone camera, you can save it as a contact.

**What to use**: `reader` · `read_order` (this notebook), regular expressions (`re`), `cv2.QRCodeEncoder`

| To find | Regular expression |
|---|---|
| Mobile phone (Korean format, 010-1234-5678) | `01[016789][-. ]?\d{3,4}[-. ]?\d{4}` |
| Email | `[\w.+-]+@[\w-]+\.[\w.]+` |

In [ ]:
import re, json

card = grab()                                     # hold a business card up to the webcam, then run
text = read_order(reader.readtext(cv2.cvtColor(card, cv2.COLOR_BGR2GRAY)), min_conf=0.3)
phones = re.findall(r"01[016789][-. ]?\d{3,4}[-. ]?\d{4}", text)   # Korean mobile number format (010-1234-5678)
emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
lines = [l.strip() for l in text.splitlines() if l.strip()]
# Name candidate: a line of 2~4 Korean (Hangul) letters, the usual length of a Korean name.
# On a card with no such line (e.g. an English card), the first line is used instead.
name = next((l for l in lines if re.fullmatch(r"[가-힣]{2,4}", l.replace(" ", ""))), lines[0] if lines else "")

contact = {"name": name, "phone": phones[:1], "email": emails[:1], "all_text": lines}
print(json.dumps(contact, ensure_ascii=False, indent=2))

mecard = f"MECARD:N:{name};" + (f"TEL:{phones[0]};" if phones else "") + (f"EMAIL:{emails[0]};" if emails else "") + ";"
q = cv2.QRCodeEncoder.create().encode(mecard)
q = cv2.copyMakeBorder(cv2.resize(q, (q.shape[1] * 8, q.shape[0] * 8), interpolation=cv2.INTER_NEAREST), 32, 32, 32, 32, cv2.BORDER_CONSTANT, value=255)
cv2.imwrite("contact_qr.png", q)
Path("contact.json").write_text(json.dumps(contact, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved: contact.json · contact_qr.png  | QR check:", [t for t, _ in read_qr(cv2.cvtColor(q, cv2.COLOR_GRAY2BGR))])

**Go further**
- Also find office phone numbers (02-…, 031-…), or phone numbers in your own country's format
- Read several business cards one after another into one CSV contact list
- Make the name guess smarter: use the biggest text (the line with the tallest rectangle) as the name